# M20b - UCI Appliances Energy, executable four-split panel

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

Four deterministic chronological splits are fixed independently of performance. Standardization is fitted on each training split only. This notebook generates the submission evidence for Appliances Energy.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import tcr_core as tcr
REPRO=ROOT/'results'/'reproduced'; REPRO.mkdir(parents=True,exist_ok=True)
def bootstrap_mean(x,n_boot=30000,seed=1):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    rng=np.random.default_rng(seed); idx=rng.integers(0,len(x),size=(n_boot,len(x)))
    b=x[idx].mean(axis=1)
    return float(x.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))
def split_starts(n,total,n_splits=4):
    starts=np.rint(np.linspace(0,n-total,n_splits)).astype(int)
    assert len(np.unique(starts))==n_splits and starts[-1]+total<=n
    return starts

def run_dataset(name,X,y,ntr,nv,nt):
    rows=[]; wins=[]; total=ntr+nv+nt
    starts=split_starts(len(X),total,4)
    for split_id,start in enumerate(starts):
        Xs=X[start:start+total].copy(); ys=y[start:start+total].copy()
        Xtr,Xv,Xt=Xs[:ntr],Xs[ntr:ntr+nv],Xs[ntr+nv:]
        ytr,yv,yt=ys[:ntr],ys[ntr:ntr+nv],ys[ntr+nv:]
        mu,sd=Xtr.mean(0),Xtr.std(0)+1e-12; Xtr=(Xtr-mu)/sd; Xv=(Xv-mu)/sd; Xt=(Xt-mu)/sd
        for path in ['temperature','gain','leak','sparsity']:
            case=tcr.evaluate_arrays(Xtr,ytr,Xv,yv,Xt,yt,name,split_id,path,20260718,50,13,1e-4,abs_tol=.002,rel_tol=.10,return_predictions=(path=='temperature'))
            plain={k:v for k,v in case.items() if k not in {'candidate_values','pred_test','y_test','safe_idxs','test_scores','train_target_scale'}}; plain['split_start']=int(start); rows.append(plain)
            if path=='temperature':
                w=tcr.window_rows(case,50,25); w.insert(0,'split_id',split_id); w.insert(1,'split_start',int(start)); wins.append(w)
    return pd.DataFrame(rows),pd.concat(wins,ignore_index=True)

def summarize(cases):
    rows=[]
    for i,(path,g) in enumerate(cases.groupby('path',sort=False)):
        sg=bootstrap_mean(g.safe_gain,seed=20260913+i); sm=bootstrap_mean(g.safe_minus_matched_gain,seed=20262913+i)
        rows.append({'path':path,'n_splits':len(g),'mean_safe_width':g.safe_width.mean(),'mean_safe_gain':sg[0],'safe_gain_ci_low':sg[1],'safe_gain_ci_high':sg[2],'mean_full_gain':g.full_gain.mean(),'mean_safe_minus_matched_gain':sm[0],'safe_minus_matched_ci_low':sm[1],'safe_minus_matched_ci_high':sm[2],'near_containment':g.near_contained.mean(),'matched_near_rate':g.matched_near_rate.mean(),'near_containment_ratio':g.near_contained.mean()/(g.matched_near_rate.mean()+1e-12),'support_range':g.support_range.mean(),'spectral_radius_range':g.spectral_radius_range.mean()})
    return pd.DataFrame(rows)

In [2]:
DATA=ROOT/'data'/'energydata_complete.csv'
if not DATA.exists(): raise FileNotFoundError('Place energydata_complete.csv in data/; see data/README.md')
df=pd.read_csv(DATA); dt=pd.to_datetime(df['date']); numeric=df.drop(columns=['date']).apply(pd.to_numeric,errors='coerce'); X=numeric.copy(); X['hour_sin']=np.sin(2*np.pi*dt.dt.hour/24); X['hour_cos']=np.cos(2*np.pi*dt.dt.hour/24); X['dow_sin']=np.sin(2*np.pi*dt.dt.dayofweek/7); X['dow_cos']=np.cos(2*np.pi*dt.dt.dayofweek/7); y=np.log1p(numeric['Appliances'].shift(-6)); X=X.iloc[:-6].reset_index(drop=True); y=y.iloc[:-6].reset_index(drop=True); assert len(X)==19729 and X.shape[1]==32
cases,wins=run_dataset('appliances',X.to_numpy(float),y.to_numpy(float),2500,1000,1000); cases.to_csv(REPRO/'m20b_replication_case_metrics.csv',index=False); wins.to_csv(REPRO/'m20b_replication_window_metrics.csv',index=False); summary=summarize(cases); summary.to_csv(REPRO/'m20b_replication_summary.csv',index=False); display(summary.round(6)); print('temperature indicator correlations:',tcr.safe_corr(wins.safe_dispersion,wins.safe_gain),tcr.safe_corr(wins.anchor_spread,wins.safe_gain))

,path,n_splits,mean_safe_width,mean_safe_gain,safe_gain_ci_low,safe_gain_ci_high,mean_full_gain,mean_safe_minus_matched_gain,safe_minus_matched_ci_low,safe_minus_matched_ci_high,near_containment,matched_near_rate,near_containment_ratio,support_range,spectral_radius_range
0,temperature,4,2.75,0.006847,0.000651,0.017349,0.042316,-0.009347,-0.032746,0.019329,0.25,0.361742,0.691099,45.690536,0.174737
1,gain,4,2.75,0.005104,0.000000,0.011808,0.016163,0.008919,-0.006427,0.024266,0.50,0.236111,2.117647,0.000000,1.473479
2,leak,4,3.75,0.003843,0.000000,0.011528,0.018695,0.035228,-0.005114,0.087742,0.75,0.297222,2.523364,0.000000,0.000000
3,sparsity,4,1.00,0.000000,0.000000,0.000000,0.042725,0.004141,-0.012980,0.021261,0.25,0.134615,1.857143,48.000000,0.167380


temperature indicator correlations: 0.6726446702696314 0.6402896720584221
